# Sesión de clase 14 · Ejercicio propuesto: extiende el asistente de la Tienda Virtual

**Objetivo:** construir una capacidad nueva del asistente reutilizando las piezas de la clase: la base
vectorial (`../bdv`), la ontología (`../ontologia`), las herramientas (`../mcp_tienda/herramientas.py`),
el MCP server y el agente robusto (`../mcp_tienda/agente.py`).

**Elige una opción:**

| Opción | Qué construyes | Piezas que reutilizas |
|---|---|---|
| **A · Asistente post-venta** | tool `clientes_insatisfechos(pais)` que cruce reseñas de entrega negativas con las órdenes en vivo del cliente y redacte un borrador de respuesta | ontología, `ordenes_cliente`, LLM |
| **B · Ontología extendida** | clase `tv:Especificacion` (desde `/api/productos/:id`) y 2 intenciones deterministas nuevas expuestas por MCP | ontología, SPARQL, SHACL, router |
| **C · Recomendador conversacional** | tool `recomendar(cliente_id)` que use lo que el cliente compró (ontología) y la similitud semántica (ChromaDB) para sugerir productos que no tiene | ontología, ChromaDB, API |

**Entregable:**
1. Este notebook ejecutado de principio a fin.
2. Tu archivo `../mcp_tienda/server_extendido.py` con la(s) tool(s) nueva(s).
3. Cinco preguntas de prueba con la traza del agente y la respuesta esperada.
4. Una comparación **con vs. sin** tu tool: ¿qué responde el agente antes y después de agregarla?
5. Una reflexión breve (5–8 líneas): ¿qué parte de tu solución es determinista y cuál generativa? ¿por qué?

**Rúbrica (20 puntos)**

| Criterio | Puntos |
|---|---|
| La tool funciona aislada (llamada directa a la función) | 4 |
| Está expuesta por MCP con nombre, docstring y type hints claros (aparece bien en `tools/list`) | 4 |
| El agente la usa correctamente en al menos 4 de las 5 preguntas de prueba | 4 |
| Comparación con vs. sin, con resultados y conclusión | 4 |
| Reflexión: determinista vs. generativo, limitaciones | 4 |

## Configuración

In [1]:
import os, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"; os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
import json, sys
from pathlib import Path

import pandas as pd
import requests

sys.path.append(".."); sys.path.insert(0, "../mcp_tienda")
import comparacion as cmp
import herramientas as h

print("Tienda:", requests.get("http://localhost:3000/api/health", timeout=5).json()["status"])
print("Tools existentes: buscar_productos, buscar_resenas, consultar_ontologia, stock_en_vivo, ordenes_cliente")

Tienda: ok
Tools existentes: buscar_productos, buscar_resenas, consultar_ontologia, stock_en_vivo, ordenes_cliente


## 1. Piezas disponibles

Recordatorio de lo que ya existe y se puede llamar directamente desde `herramientas`:

In [2]:
# Consultas SPARQL sobre la ontología (con parámetros enlazados de forma segura)
h._sparql("""
SELECT ?nombre (COUNT(?o) AS ?ordenes) WHERE { ?o a tv:Orden ; tv:contiene ?p . ?p tv:nombre ?nombre }
GROUP BY ?nombre ORDER BY DESC(?ordenes) LIMIT 5
""")

[{'nombre': 'Nintendo Switch Lite', 'ordenes': 11},
 {'nombre': 'Canon EOS Rebel T7', 'ordenes': 11},
 {'nombre': 'Logitech MX Master 3S', 'ordenes': 11},
 {'nombre': 'Sennheiser Momentum 4', 'ordenes': 11},
 {'nombre': 'Razer DeathAdder V3', 'ordenes': 11}]

In [3]:
# Búsqueda semántica, entidades y API en vivo
print(h.buscar_productos("cámara para viajes", precio_max=3000)[:2])
print(h.resolver_producto("la Nikon"))
print(h.ordenes_cliente(35)["total_ordenes"], "órdenes del cliente 35")

W1122 14:11:07.461000 10682 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


[{'nombre': 'Insta360 X4', 'categoria': 'Fotografia y Video', 'subcategoria': 'Camaras de Accion', 'precio': 2299, 'calificacion': 5.0, 'similitud': 0.56}, {'nombre': 'Canon EOS Rebel T7', 'categoria': 'Fotografia y Video', 'subcategoria': 'Camaras DSLR', 'precio': 1899, 'calificacion': 2.8, 'similitud': 0.557}]
None
2 órdenes del cliente 35


## 2. Implementa la lógica

Escribe aquí la función de tu opción. Pruébala sola antes de exponerla por MCP. Las tres opciones
tienen un esqueleto; borra las que no uses.

In [4]:
# Opción A · Asistente post-venta
def clientes_insatisfechos(pais: str) -> list[dict]:
    """Clientes de `pais` con reseñas de entrega de 1–2★: producto, texto de la reseña y sus órdenes recientes."""
    # TODO 1: consulta SPARQL → clientes (id) con reseñas tv:ResenaEntrega de calificación <= 2 que residen en `pais`
    # TODO 2: para cada cliente, h.ordenes_cliente(cliente_id) → número de órdenes y total gastado
    # TODO 3: ordenar por total gastado (los clientes más valiosos primero)
    return []

clientes_insatisfechos("Chile")

[]

In [5]:
# Opción B · Ontología extendida
def especificaciones_en_vivo(producto: str) -> list[dict]:
    """Especificaciones (clave/valor) de un producto desde GET /api/productos/:id."""
    # TODO 1: h.resolver_producto(producto) → id
    # TODO 2: GET /api/productos/{id} con x-api-key → campo "especificaciones"
    # TODO 3 (en una celda aparte): agregar tv:Especificacion, tv:tieneEspecificacion, tv:clave, tv:valor
    #         al grafo, una forma SHACL para ellas y dos intenciones nuevas (p. ej. "productos de la marca X",
    #         "productos con autonomía mayor a N horas")
    return []

especificaciones_en_vivo("DJI Air 3")

[]

In [6]:
# Opción C · Recomendador conversacional
def recomendar(cliente_id: int, k: int = 3) -> list[dict]:
    """Productos que el cliente no ha comprado, parecidos a lo que sí compró."""
    # TODO 1: SPARQL → productos que compró el cliente (tv:Orden tv:realizadaPor / tv:contiene)
    # TODO 2: buscar en ChromaDB productos parecidos a cada uno (h.buscar_productos con su nombre)
    # TODO 3: quitar los ya comprados, agregar el stock en vivo (h.stock_en_vivo) y quitar los agotados
    return []

recomendar(35)

[]

## 3. Expón la tool por MCP

No hace falta modificar `server.py`: `server_extendido.py` importa el server base y le agrega tools.
Así las 5 tools de la clase siguen disponibles.

In [7]:
%%writefile ../mcp_tienda/server_extendido.py
"""MCP server de la Tienda Virtual + las tools del ejercicio propuesto."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import herramientas as h
from server import mcp          # el server base, con sus 5 tools, resources y prompt


@mcp.tool()
def mi_tool_nueva(parametro: str) -> dict:
    """TODO: describe qué hace y cuándo usarla (es lo que lee el LLM para decidir)."""
    # TODO: llamar aquí a tu función (copia su código a herramientas.py o a este archivo)
    return {"pendiente": parametro}


if __name__ == "__main__":
    mcp.run()

Writing ../mcp_tienda/server_extendido.py


In [8]:
# Verifica que tu tool aparece en tools/list (se usa un cliente MCP igual que en el Ejercicio 3)
from agente import AgenteTienda, mostrar_traza

class SoloMCP(AgenteTienda):        # el agente sin LLM, solo para listar y llamar tools
    def __init__(self):
        super().__init__(llm=None, tokenizer=None, servidor=Path("../mcp_tienda/server_extendido.py"))

conexion = SoloMCP()
print(await conexion.conectar())

['buscar_productos', 'buscar_resenas', 'consultar_ontologia', 'stock_en_vivo', 'ordenes_cliente', 'mi_tool_nueva']


In [9]:
from agente import resultado_tool
resultado_tool(await conexion.cliente.call_tool("mi_tool_nueva", {"parametro": "prueba"}))

{'pendiente': 'prueba'}

## 4. Conecta tu tool al agente

Para que el agente robusto la use tienes dos caminos (elige uno y justifícalo en la reflexión):

- **Determinista:** agrega una intención en `../ontologia/intenciones.json` con frases de ejemplo y
  mapea esa intención a tu tool en `INTENCION_A_TOOL` (`agente.py`).
- **Generativo:** agrega tu tool a la lista `opciones` que ve el LLM y un ejemplo *few-shot* nuevo.

In [10]:
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
transformers.logging.set_verbosity_error()

MODELO_LLM = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODELO_LLM)
llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, dtype=torch.float32); llm.eval()

# TODO: cuando tu tool esté conectada, usa servidor=Path("../mcp_tienda/server_extendido.py")
agente = AgenteTienda(llm, tokenizer, estrategia="robusta")
await agente.conectar()

['buscar_productos',
 'buscar_resenas',
 'consultar_ontologia',
 'stock_en_vivo',
 'ordenes_cliente']

## 5. Preguntas de prueba y comparación con vs. sin tu tool

In [11]:
MIS_PREGUNTAS = pd.DataFrame([
    # TODO: 5 preguntas que tu tool debería resolver, con los términos esperados separados por "|"
    {"id": 1, "tipo": "propuesta", "pregunta": "¿Qué clientes de Chile están insatisfechos con la entrega?", "esperado": "TODO"},
])

respuestas_sin = []
for p in MIS_PREGUNTAS.pregunta:
    r = await agente.preguntar(p)
    mostrar_traza(r)
    respuestas_sin.append(r)

res_sin = cmp.registrar("propuesto_sin_tool", [r["respuesta"] for r in respuestas_sin],
                        [r["latencia_s"] for r in respuestas_sin], banco=MIS_PREGUNTAS)
# TODO: repetir con el agente conectado a server_extendido.py → res_con = cmp.registrar("propuesto_con_tool", ...)
# cmp.lado_a_lado(res_sin, res_con)
res_sin[["pregunta", "respuesta", "resultado"]]

❓ ¿Qué clientes de Chile están insatisfechos con la entrega?
   · router               {"intencion": "resenas_negativas_pais", "confianza": 0.75}
   · call_tool            {"tool": "consultar_ontologia", "args": {"pregunta": "¿Qué clientes de Chile están insatisfechos con la entrega?"}, "error_mcp": false, "resultado": "{'determinista': True, 'intenc
🤖 Productos con reseñas de entrega negativas (≤ 2★) de clientes de Chile (4): Garmin Forerunner 265; HP Pavilion 15 Intel Core i5; OnePlus 12 256GB; Sennheiser Momentum 4.  (5.4 s)



,pregunta,respuesta,resultado
0,¿Qué clientes de Chile están insatisfechos con...,Productos con reseñas de entrega negativas (≤ ...,incorrecta


In [12]:
await agente.cerrar()
await conexion.cerrar()

## 6. Reflexión

*TODO: escribe aquí 5–8 líneas.* ¿Qué parte de tu solución es determinista y cuál generativa? ¿Qué
preguntas resolvería mal tu tool? ¿Qué cambiaría si el modelo fuera mucho más grande?